In [8]:
import cv2
import mediapipe as mp
import warnings
warnings.filterwarnings("ignore")

# MediaPipe의 그리기 도구, 스타일, 손 인식 기능 가져오기
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

# 영상 파일 열기
cap = cv2.VideoCapture('hands.mp4')

# 손 인식 객체 생성
with mp_hands.Hands(
    model_complexity=0,              # 가벼운 모델 사용(속도 우선)
    min_detection_confidence=0.5,    # 손을 처음 검출할 최소 신뢰도
    min_tracking_confidence=0.5      # 검출된 손을 추적할 최소 신뢰도
) as hands:
    
drawing_mode = False
prev = None

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        # 현재 프레임 크기 구한 뒤 절반으로 축소(속도 향상 목적)
        h, w, _ = image.shape
        image = cv2.resize(image, (w // 2, h // 2))

        # 성능 향상을 위해 일단 이미지 수정 불가로 설정
        image.flags.writeable = False

        # OpenCV는 BGR, MediaPipe는 RGB를 사용하므로 색상 순서 변환
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # RGB 이미지에서 손 검출 및 랜드마크 추출
        results = hands.process(rgb_image)
        image.flags.writeable = True


        # 손이 검출된 경우
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # 검지끝, 검지중간, 손목 좌표 검출 
                landmarks = hand_landmarks.landmark
                tip = landmarks[mp_hands.HandLandmark.INDEX_FINGER_TIP]
                pip = landmarks[mp_hands.HandLandmark.INDEX_FINGER_PIP]
                wrist = landmarks[mp_hands.HandLandmark.WRIST]

                tip_x = int(tip.x * w)
                tip_y = int(tip.y * h)
                pip_x = int(pip.x * w)
                pip_y = int(pip.y * h)
                wrist_x = int(wrist.x * w)
                wrist_y = int(wrist.y * h)
                distance = ((tip_x - wrist_x)**2 + (tip_y - wrist_y)**2) * 0.5 
                    
                # 검지 펼침 여부 & 화면 가까움 여부
                if tip_y > pip_y and distance > 50 :
                    drawing_mode = True
                    else :
                    drawing_mode = False
                    prev = None

                if drawing_mode:
                    if prev:
                        cv2.line(블라블라)
                        prev = now
            

                
                # 손의 랜드마크 점과 연결선을 현재 프레임에 그림
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style()
                )

        cv2.imshow('Hands', image)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

367 76
369 73
369 65
372 65
374 61
372 59
373 57
380 53
387 53
389 51
391 57
392 59
389 60
392 60
392 67
391 67
387 65
388 65
385 67
382 66
380 66
377 68
377 71
373 70
373 65
374 67
374 66
373 69
367 64
369 59
370 56
364 55
358 46
362 44
365 45
367 43
367 45
370 42
369 45
369 45
368 48
364 47
361 45
365 51
362 46
368 50
372 51
375 48
379 44
375 49
386 47
385 41
407 26
414 32
407 44
423 52
427 66
432 82
433 105
444 132
417 184
376 261
377 286
372 294
371 319
355 346
361 352
367 357
360 370
355 367
359 375
363 376
359 374
352 377
335 367
331 353
337 355
333 326
360 294
391 259
425 171
443 95
432 59
421 60
414 93
428 112
425 120
424 121
431 131
441 132
447 131
450 108
463 103
455 90
459 47
490 -32
492 -30
483 5
500 10
501 38
504 57
497 63
498 53
498 42
486 28
480 35
477 46
474 60
483 67
474 95
482 111
484 167
467 256
465 248
517 164
484 179
505 208
498 225
480 235
489 269
463 303
437 307
426 313
423 325
424 327
419 328
416 334
415 336
418 336
415 338
417 340
417 341
417 343
418 344
428 32

In [ ]:
이전좌표 = None
그리기모드 = False

while 영상반복:
    프레임읽기
    손검출

    if 손이검출되면:
        검지끝 좌표 구하기
        검지 펼침 여부 확인
        손이 가까운지 확인

        if 검지 펼침 and 손 가까움:
            그리기모드 = True
        else:
            그리기모드 = False
            이전좌표 = None

        if 그리기모드:
            if 이전좌표가 존재하면:
                이전좌표 ~ 현재좌표 사이에 선 그리기
            이전좌표 = 현재좌표

In [6]:
import cv2
import mediapipe as mp
import warnings
import numpy as np
warnings.filterwarnings("ignore")

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture('hand.mp4')

drawing_mode = False
canvas = None
prev = None

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        h, w, _ = image.shape
        image = cv2.resize(image, (w // 2, h // 2))
        h, w, _ = image.shape

        if canvas is None:
            canvas = np.zeros_like(image)

        image.flags.writeable = False
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)
        image.flags.writeable = True

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                landmarks = hand_landmarks.landmark

                tip = landmarks[mp_hands.HandLandmark.INDEX_FINGER_TIP]
                pip = landmarks[mp_hands.HandLandmark.INDEX_FINGER_PIP]
                wrist = landmarks[mp_hands.HandLandmark.WRIST]
                mcp = landmarks[mp_hands.HandLandmark.MIDDLE_FINGER_MCP]
                pinky_tip = landmarks[mp_hands.HandLandmark.PINKY_TIP]
                pinky_pip = landmarks[mp_hands.HandLandmark.PINKY_PIP]
                pinky_mcp = landmarks[mp_hands.HandLandmark.PINKY_MCP]

                tip_x = int(tip.x * w)
                tip_y = int(tip.y * h)
                pip_x = int(pip.x * w)
                pip_y = int(pip.y * h)
                wrist_x = int(wrist.x * w)
                wrist_y = int(wrist.y * h)
                mcp_x = int(mcp.x * w)
                mcp_y = int(mcp.y * h)
                pink_tip_x = int(pinky_tip.x * w)
                pink_tip_y = int(pinky_tip.y * h)
                pink_pip_x = int(pinky_pip.x * w)
                pink_pip_y = int(pinky_pip.y * h)
                pink_mcp_x = int(pinky_mcp.x * w)
                pink_mcp_y = int(pinky_mcp.y * h)
                
                now = (tip_x, tip_y)
                now_pinky = (pink_tip_x, pink_tip_y)
                distance = ((tip_x - wrist_x) ** 2 + (tip_y - wrist_y) ** 2) ** 0.5

                index_open = tip_y < pip_y
                index_close = tip_y > pip_y
                pinky_open = (pink_tip_y < pink_pip_y) and (pink_pip_y < pink_mcp_y)
                close_enough = distance > 250

                # if pinky_open:
                #     cv2.rectangle(image, (50, 50), now_pinky, (0, 0, 255), -1)
                if pinky_open:
                    cv2.putText(image, 'ERASE', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                
                if not drawing_mode:
                    if index_open and close_enough:
                        drawing_mode = True
                        prev = now
                
                else: 
                    if not index_open:
                        drawing_mode = False
                        prev = None
                    else:
                        if prev is not None:
                            cv2.line(canvas, prev, now, (255, 200, 100), 5)
                            prev = now

                print(distance)
                
                # mp_drawing.draw_landmarks(
                #     image,
                #     hand_landmarks,
                #     mp_hands.HAND_CONNECTIONS,
                #     mp_drawing_styles.get_default_hand_landmarks_style(),
                #     mp_drawing_styles.get_default_hand_connections_style()
                # )
        else:
            prev = None

        result = cv2.add(image, canvas)

        cv2.imshow('Hands', result)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

87.28115489611719
83.72574275573791
90.79647570252934
90.6697303403953
90.79647570252934
91.7877987534291
90.79647570252934
89.94442728707543
90.79647570252934
87.82368700982668
89.94442728707543
89.80534505250787
90.93404203047393
88.81441324469807
90.6697303403953
88.68483523128404
88.81441324469807
86.70063436907483
88.68483523128404
86.57944328765345
84.48076704197234
82.38931969618392
82.29823813423954
81.39410298049853
81.49846624323675
81.30190649671138
81.61494961096282
82.38931969618392
80.30566605165541
81.74350127074322
81.39410298049853
82.60750571225353
82.49242389456137
80.39900496896712
81.49846624323675
82.49242389456137
80.30566605165541
80.50465825031493
80.39900496896712
80.62257748298549
83.60023923410746
81.61494961096282
81.74350127074322
83.02409288875127
82.03657720797473
80.23091673413684
80.06247560499239
78.91767862779544
79.90619500389191
78.91767862779544
78.91767862779544
79.76214641043707
78.77182237323191
78.5175139698144
79.76214641043707
78.51751396981